## Setup

First, let's install the required packages and set our API keys

In [ ]:
%%capture --no-stderr
%pip install -U langchain_openai langgraph langgraph-checkpoint-redis

In [ ]:
import getpass
import os

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

## Define the vector store

This vector store will be used lately by the graph to store and retrieve memories across threads

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langgraph.store.redis import RedisStore
from langgraph.store.base import IndexConfig

index_config: IndexConfig = {
    "dims": 1536,
    "embed": OpenAIEmbeddings(model="text-embedding-3-small"),
    "ann_index_config": {
        "vector_type": "vector",
    },
    "distance_type": "cosine",
}

REDIS_URI = "redis://localhost:6379"
redis_store = None
with RedisStore.from_conn_string(REDIS_URI, index=index_config) as _redis_store:
    _redis_store.setup()
    redis_store = _redis_store

## Create implementation

Let's implement the graph that will leverage Redis for storing state transitions and user data.

In [ ]:
import uuid
from IPython.display import Image, display
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.redis import RedisSaver
from langgraph.store.base import BaseStore

model = ChatOpenAI(model="gpt-4o", temperature=0)

def call_model(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    user_id = config["configurable"]["user_id"]
    namespace = ("memories", user_id)
    memories = store.search(namespace, query=str(state["messages"][-1].content))
    info = "\n".join([d.value["data"] for d in memories])
    system_msg = f"You are a helpful assistant talking to the user. User info: {info}"

    last_message = state["messages"][-1]
    if "remember" in last_message.content.lower():
        memory = "User name is Bob"
        store.put(namespace, str(uuid.uuid4()), {"data": memory})

    response = model.invoke(
        [{"role": "system", "content": system_msg}] + state["messages"]
    )
    return {"messages": response}

builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

REDIS_URI = "redis://localhost:6379"
checkpointer = None
with RedisSaver.from_conn_string(REDIS_URI) as _checkpointer:
    _checkpointer.setup()
    checkpointer = _checkpointer

graph = builder.compile(checkpointer=checkpointer, store=redis_store)
display(Image(graph.get_graph().draw_mermaid_png()))

## Usage
Let's see multiple users using the graph

In [ ]:
def print_stream(stream):
    for s in stream:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

Let's interact with the first user to get their name memorized

In [ ]:
config = {"configurable": {"thread_id": "1", "user_id": "1"}}
input_message = {"role": "user", "content": "Hi! Remember: my name is Bob"}
print_stream(graph.stream({"messages": [input_message]}, config, stream_mode="values"))

Now let's check if the name is indeed part of the memory

In [ ]:
config = {"configurable": {"thread_id": "2", "user_id": "1"}}
input_message = {"role": "user", "content": "what is my name?"}
print_stream(graph.stream({"messages": [input_message]}, config, stream_mode="values"))

Let's now run the graph for another user to verify that the memories about the first user are self contained:

In [ ]:
config = {"configurable": {"thread_id": "3", "user_id": "2"}}
input_message = {"role": "user", "content": "what is my name?"}
print_stream(graph.stream({"messages": [input_message]}, config, stream_mode="values"))

Optionally, you can inspect the Redis store to verify the saved memories:

In [ ]:
for memory in redis_store.search(("memories", "1")):
    print(memory.value)